# Scenario Playground Step 7: State Features

Objective: condition every selected event-aware row on the macro, market, risk, and trend information available at its completed pre-shock `state_cutoff_date`.

Success means 25-60 disciplined features, complete source provenance, no future release or market observation, and no target, shock, split, or model construction.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "scenario_modeling").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Run this notebook from inside the InsightPulse repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scenario_modeling.src.state_features import (
    MACRO_STATE_FEATURES,
    MARKET_DERIVED_FEATURES,
    STATE_FEATURE_NAMES,
    build_state_artifacts,
    persist_state_artifacts,
)

artifacts = build_state_artifacts()
output_paths = persist_state_artifacts(artifacts)
features = artifacts["features"]
availability = artifacts["availability_audit"]
summary = artifacts["summary"]
print({"rows": len(features), "state_features": len(STATE_FEATURE_NAMES), "macro": len(MACRO_STATE_FEATURES), "market_risk_trend": len(MARKET_DERIVED_FEATURES)})
output_paths

## Feature families and formulas

Macro releases use conservative availability dates. Rolling market features use only the source history ending at or before the row's state cutoff.

In [ ]:
definition_rows = []
for feature, definition in summary["definitions"].items():
    family = "macro" if feature in MACRO_STATE_FEATURES else ("market_level" if feature == "vix_level_points" else "market_risk_trend")
    definition_rows.append({"family": family, "feature": feature, **definition})
definitions = pd.DataFrame(definition_rows)
print(definitions.groupby("family").size().rename("feature_count"))
definitions

## Completeness and ranges

No imputation, clipping, winsorization, normalization, or full-sample fitting occurs in this step.

In [ ]:
missingness = pd.Series(summary["missingness"], name="missing_rows")
range_table = pd.DataFrame(summary["distributions"]).T[
    ["count", "missing_count", "minimum", "median", "maximum", "mean", "standard_deviation"]
]
print("Total missing state values:", int(missingness.sum()))
range_table

## Point-in-time availability

The audit retains the date that made each macro value usable and the exact end date used by every rolling market family.

In [ ]:
pit_checks = pd.Series(summary["point_in_time_by_availability_column"], name="on_or_before_cutoff")
print("Availability columns checked:", len(pit_checks))
print("All pass:", bool(pit_checks.all()))
print("\nMacro staleness (normal for monthly and quarterly releases):")
print(pd.DataFrame(summary["macro_staleness"]).T)
assert pit_checks.all()
availability.head(3)

## Market diagnostics

The cached `CL=F` history contains one nonpositive 2020 close. Its untrimmed effect is retained for Step 9's outlier audit rather than hidden during feature construction.

In [ ]:
market_diagnostics = pd.DataFrame(summary["market_diagnostics"]).T
market_diagnostics[["source", "field", "maximum_source_age_calendar_days", "nonpositive_raw_value_count"]].fillna(0)

## Step 7 acceptance decision

The output remains an independent state-feature family. It is not joined to Step 6 shocks or Step 5 labels here.

In [ ]:
validation = pd.Series(summary["validations"], name="passed")
print(validation.to_string())
assert validation.all()
assert len(features) == 585
assert len(STATE_FEATURE_NAMES) == 46
assert features[list(STATE_FEATURE_NAMES)].isna().sum().sum() == 0
print("STEP_7_COMPLETE")